In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O
from src.feature_engineering import *

In [ ]:
# Loading clean dataset

df = pd.read_csv("../data/processed/merged_cleaned.csv")

print(df.shape)
df.head()

In [ ]:
# student level dataset
student_df = build_student_features(df)

student_df.head()

In [ ]:
# output
student_df.info()

In [ ]:
# basic check 
 #Did aggregation break anything?
 #Any missing values introduced?
 # Are all types correct?
student_df.isnull().sum()

In [ ]:
# adding advance features (maybe they wont be too useful)
# Early behavior → future completion : students at risk
early_engagement = (
    df[df["day"] <= 3]
    .groupby("student_id")["engagement_score"]
    .mean()
)

student_df["early_engagement"] = student_df["student_id"].map(early_engagement)
# total active days
 # Measure persistence
# Help detect dropout


total_days = df.groupby("student_id")["day"].count()

student_df["total_days"] = student_df["student_id"].map(total_days)

# engagement trend
# Are students improving or declining
# signal for retention
engagement_trend = (
    df.groupby("student_id")
    .apply(lambda x: x["engagement_score"].iloc[-1] - x["engagement_score"].iloc[0])
)

student_df["engagement_trend"] = student_df["student_id"].map(engagement_trend)

In [ ]:
# sanity check

# values outside expected ranges
# weird distributions

student_df.describe()

In [ ]:
# feature insights
# what drives completion?
# where are inequalities?
# what should we fix?

# completion vs attendance
student_df.groupby("completion")["attendance_rate"].mean()

#early engagement vs completion
student_df.groupby("completion")["early_engagement"].mean()

# disability vs engagement
student_df.groupby("has_disability")["avg_engagement"].mean()

In [ ]:
# handling missing values
# Some students may drop early or  not have day 1 to day 3 data
student_df["early_engagement"] = student_df["early_engagement"].fillna(
    student_df["avg_engagement"]
)

In [ ]:
# saving process dataset
student_df.to_csv("../data/processed/student_level.csv", index=False)

In [ ]:
# metrics standarisation

METRIC_DEFINITIONS = {
    "completion_rate": "Proportion of students who completed the course",
    "engagement_index": "Average engagement normalized to [0,1]",
    "learning_gain": "Self-reported learning weighted by attendance",
    "equity_gap": "Difference between best and worst performing groups"
}